In [ ]:
import os
import re
import numpy as np
import tensorflow as tf
from PIL import Image
import io
def load_weather_data(weather_txt_path):
    """
    weather.txt에서 각 줄을 읽어 float 리스트로 반환.
    예: 한 줄에 "81.0 14.0 0.0 12.0 224.0 0.0" -> [81.0, 14.0, 0.0, 12.0, 224.0, 0.0]
    """
    weather_data = []
    with open(weather_txt_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            vals = line.strip().split()
            vals = list(map(float, vals))
            weather_data.append(vals)
    return weather_data

def load_image_as_bytes(image_path, mode='L', size=None, target = None):
    """
    
    Pillow로 이미지를 로드:
      - mode='L': Grayscale (1채널)
      - mode='RGB': 컬러 (3채널)
    - (옵션) size=(W,H) 리사이즈
    - 0~1 범위(float32)로 정규화한 뒤 np.array -> bytes
    """
    with Image.open(image_path) as img:
        img = img.convert(mode)
        if size is not None:
            img = img.resize(size)
        if target == 'per':

            img_np_12h = np.array(img, dtype=np.float32)
            imgs=[(img_np_12h/255).tobytes()]
            cuts= [238.75,222.5,206.25,190,173.75,157.5,141.25,125,108.75,92.5,76.25]
            for i in range(11) :
                temp = img_np_12h.copy()
                temp[temp < cuts[10-i]] = 0
                imgs.append((temp/255).tobytes())


            return imgs
        elif target == 'ter':
            img_np_12h = np.array(img, dtype=np.float32)/ 255

            return img_np_12h.tobytes()

        
    


# TFRecord Feature 변환 유틸
def _bytes_feature(value: bytes):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _float_bytes_feature(array: np.ndarray):
    """
    float32 numpy array를 raw bytes로 변환하여 bytes feature로 만듦.
    array.shape는 (H,W,C) 또는 (N,) 등 다양한 형태여도 가능.
    """
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[array.tobytes()]))

def write_tfrecord(
    output_path,
    weather_txt_path,
    spread_dir,
    terrain_dir,
    dem_dir,
    fuel_dir,
    spread_size=(128,128),    # 확산이미지 리사이즈 (Grayscale)
    terrain_size=(128,128),   # 지형이미지 리사이즈 (RGB)
    weather_size=(128,128)    # 날씨데이터를 이 크기로 브로드캐스트
):
    """
    1) spread_dir에서 "at_(\d+)_(\d+).png" 패턴 -> (ignite, weather) 추출
    2) terrain_{ignite}.png (RGB)
    3) weather.txt에서 (temp=0, hume=1, ws=3, wd=4)
       -> temp, hume, ws는 (256,256)로 브로드캐스트
       -> wd는 sin/cos 구한 뒤 각각 (256,256) 브로드캐스트
    4) TFRecord에 저장
    """
    # 1) weather 데이터 전체 불러오기
    weather_data = load_weather_data(weather_txt_path)  # ex: 길이 N, 각 row=[temp, hume, ???, ws, wd, ???]

    # 2) spread_dir 내 파일 패턴
    pattern = re.compile(r'^at_(\d+)_(\d+)\.png$')
    file_list = sorted(os.listdir(spread_dir))

    W, H = weather_size
    with tf.io.TFRecordWriter(output_path) as writer:
        sample_count = 0

        for filename in file_list:
            m = pattern.match(filename)
            if not m:
                continue
            ignite_str, weather_str = m.groups()
            ignite_index = int(ignite_str)
            weather_index = int(weather_str)
            # spread image (Grayscale)
            spread_path = os.path.join(spread_dir, filename)
            try:
                (spread_12h_bytes, spread_11h_bytes, spread_10h_bytes, spread_9h_bytes, spread_8h_bytes, spread_7h_bytes, spread_6h_bytes, spread_5h_bytes, spread_4h_bytes,spread_3h_bytes,spread_2h_bytes, spread_1h_bytes) = load_image_as_bytes(spread_path, mode='L', size=spread_size, target='per')
            except Exception as e:
                print(f"[WARN] fail to load spread: {spread_path}, {e}")
                continue

            # terrain image (RGB)
            terrain_filename = f"ter_{ignite_index}.png"
            terrain_path = os.path.join(terrain_dir, terrain_filename)
            if not os.path.exists(terrain_path):
                print(f"[WARN] terrain_{ignite_index}.png not found, skip.")
                continue
            try:
                terrain_bytes = load_image_as_bytes(terrain_path, mode='RGB', size=terrain_size, target='ter')
            except Exception as e:
                print(f"[WARN] fail to load terrain: {terrain_path}, {e}")
                continue
            
            # dem image (RGB)
            dem_filename = f"dem_{ignite_index}.png"
            dem_path = os.path.join(dem_dir, dem_filename)
            if not os.path.exists(dem_path):
                print(f"[WARN] dem_{ignite_index}.png not found, skip.")
                continue
            try:
                dem_bytes = load_image_as_bytes(
                    dem_path, mode='L', size=spread_size, target='ter'
                )
            except Exception as e:
                print(f"[WARN] fail to load dem: {dem_path}, {e}")
                continue


            # fuel image (RGB)
            fuel_filename = f"fuel_{ignite_index}.png"
            fuel_path = os.path.join(fuel_dir, fuel_filename)
            if not os.path.exists(fuel_path):
                print(f"[WARN] fuel_{ignite_index}.png not found, skip.")
                continue
            try:
                fuel_bytes = load_image_as_bytes(
                    fuel_path, mode='L', size=spread_size, target='ter'
                )
            except Exception as e:
                print(f"[WARN] fail to load fuel: {dem_path}, {e}")
                continue

            # weather array
            if weather_index - 1 < 0 or weather_index - 1 >= len(weather_data):
                print(f"[WARN] weather_index out of range: {weather_index}")
                continue
            w = weather_data[weather_index - 1]



            spread_0hr_fake = np.zeros((128,128,1), dtype=np.float32)
            spread_0hr_fake = spread_0hr_fake.tobytes()

            # 원본 값
            temp_0h_val = w[0] 
            hume_0h_val = w[1] 
            ws_0h_val   = w[2]  
            wd_0h_val   = w[3]

            temp_1h_val = w[4] 
            hume_1h_val = w[5] 
            ws_1h_val   = w[6]  
            wd_1h_val   = w[7]

            temp_2h_val = w[8] 
            hume_2h_val = w[9] 
            ws_2h_val   = w[10]  
            wd_2h_val   = w[11]

            temp_3h_val = w[12] 
            hume_3h_val = w[13] 
            ws_3h_val   = w[14]  
            wd_3h_val   = w[15]

            temp_4h_val = w[16] 
            hume_4h_val = w[17] 
            ws_4h_val   = w[18]  
            wd_4h_val   = w[19]

            temp_5h_val = w[20] 
            hume_5h_val = w[21] 
            ws_5h_val   = w[22]  
            wd_5h_val   = w[23]

            temp_6h_val = w[24] 
            hume_6h_val = w[25] 
            ws_6h_val   = w[26]  
            wd_6h_val   = w[27]

            temp_7h_val = w[28] 
            hume_7h_val = w[29] 
            ws_7h_val   = w[30]  
            wd_7h_val   = w[31]

            temp_8h_val = w[32] 
            hume_8h_val = w[33] 
            ws_8h_val   = w[34]  
            wd_8h_val   = w[35]

            temp_9h_val = w[36] 
            hume_9h_val = w[37] 
            ws_9h_val   = w[38]  
            wd_9h_val   = w[39]

            temp_10h_val = w[40] 
            hume_10h_val = w[41] 
            ws_10h_val   = w[42]  
            wd_10h_val   = w[43] 

            temp_11h_val = w[44] 
            hume_11h_val = w[45] 
            ws_11h_val   = w[46]  
            wd_11h_val   = w[47]

            temp_hume_t1 = np.array([
    [temp_0h_val, hume_0h_val],
    [temp_1h_val, hume_1h_val],
    [temp_2h_val, hume_2h_val],
    [temp_3h_val, hume_3h_val]
], dtype=np.float32)
            temp_hume_t2 = np.array([
    [temp_4h_val, hume_4h_val],
    [temp_5h_val, hume_5h_val],
    [temp_6h_val, hume_6h_val],
    [temp_7h_val, hume_7h_val]
], dtype=np.float32)
            
            temp_hume_t3 = np.array([
    [temp_8h_val, hume_8h_val],
    [temp_9h_val, hume_9h_val],
    [temp_10h_val, hume_10h_val],
    [temp_11h_val, hume_11h_val]
], dtype=np.float32)

            tnh = np.array([
                [temp_0h_val, hume_0h_val],
    [temp_1h_val, hume_1h_val],
    [temp_2h_val, hume_2h_val],
    [temp_3h_val, hume_3h_val],
    [temp_4h_val, hume_4h_val],
    [temp_5h_val, hume_5h_val],
    [temp_6h_val, hume_6h_val],
    [temp_7h_val, hume_7h_val],
    [temp_8h_val, hume_8h_val],
    [temp_9h_val, hume_9h_val],
    [temp_10h_val, hume_10h_val],
    [temp_11h_val, hume_11h_val]
            ], dtype=np.float32)

            ws_wd_t1 = np.array([
    [ws_0h_val, wd_0h_val],
    [ws_1h_val, wd_1h_val],
    [ws_2h_val, wd_2h_val],
    [ws_3h_val, wd_3h_val]
], dtype=np.float32)
            
            ws_wd_t2 = np.array([
    [ws_4h_val, wd_4h_val],
    [ws_5h_val, wd_5h_val],
    [ws_6h_val, wd_6h_val],
    [ws_7h_val, wd_7h_val]
], dtype=np.float32)
            
            ws_wd_t3 = np.array([
    [ws_8h_val, wd_8h_val],
    [ws_9h_val, wd_9h_val],
    [ws_10h_val, wd_10h_val],
    [ws_11h_val, wd_11h_val]
], dtype=np.float32)
            
            wind = np.array([
                [ws_0h_val, wd_0h_val],
    [ws_1h_val, wd_1h_val],
    [ws_2h_val, wd_2h_val],
    [ws_3h_val, wd_3h_val],
     [ws_4h_val, wd_4h_val],
    [ws_5h_val, wd_5h_val],
    [ws_6h_val, wd_6h_val],
    [ws_7h_val, wd_7h_val],
    [ws_8h_val, wd_8h_val],
    [ws_9h_val, wd_9h_val],
    [ws_10h_val, wd_10h_val],
    [ws_11h_val, wd_11h_val]

            ], dtype=np.float32)

            # TFRecord Feature 생성
            feature_dict = {
                #'spread_6h_image':   _bytes_feature(spread_6h_bytes),
                'prev_fire' :  _bytes_feature(spread_0hr_fake),
                'fire_1' :  _bytes_feature(spread_1h_bytes),
                'fire_2' :  _bytes_feature(spread_2h_bytes),
                'fire_3' :  _bytes_feature(spread_3h_bytes),
                'fire_4' :  _bytes_feature(spread_4h_bytes),
                'fire_5' :  _bytes_feature(spread_5h_bytes),
                'fire_6' :  _bytes_feature(spread_6h_bytes),
                'fire_7' :  _bytes_feature(spread_7h_bytes),
                'fire_8' :  _bytes_feature(spread_8h_bytes),
                'fire_9' :  _bytes_feature(spread_9h_bytes),
                'fire_10' :  _bytes_feature(spread_10h_bytes),
                'fire_11' :  _bytes_feature(spread_11h_bytes),
                'fire_12' :  _bytes_feature(spread_12h_bytes),
                'terrain_image':  _bytes_feature(terrain_bytes),
                'dem_image':     tf.train.Feature(bytes_list=tf.train.BytesList(value=[dem_bytes])),
                'fuel_image':     tf.train.Feature(bytes_list=tf.train.BytesList(value=[fuel_bytes])),
                'temp':_float_bytes_feature(tnh),
                'wind':_float_bytes_feature(wind),
                # 이하 weather (모두 (256,256) shape의 단일채널)
                #'temp_hume_t1':   _float_bytes_feature(temp_hume_t1),
                #'temp_hume_t2':   _float_bytes_feature(temp_hume_t2),
                #'temp_hume_t3':   _float_bytes_feature(temp_hume_t3),
                #'ws_wd_t1':   _float_bytes_feature(ws_wd_t1),
                #'ws_wd_t2':   _float_bytes_feature(ws_wd_t2),
                #'ws_wd_t3':   _float_bytes_feature(ws_wd_t3),
                'ignite_index': tf.train.Feature(int64_list=tf.train.Int64List(value=[ignite_index])),
                'weather_index': tf.train.Feature(int64_list=tf.train.Int64List(value=[weather_index])),
            }

            example = tf.train.Example(features=tf.train.Features(feature=feature_dict))
            writer.write(example.SerializeToString())
            sample_count += 1

        print(f"[INFO] TFRecord saved: {output_path}, total {sample_count} samples")



In [ ]:
# 0) 사용자 환경에 맞게 경로 지정
weather_txt_path = "./weather copy.txt"           # 기상 데이터 txt
spread_dir       = "./n_per_fine"         # "at_{ignite}_{weather}.png" 모음
terrain_dir      = "./n_ter_fine"        # "terrain_{ignite}.png" 모음
dem_dir = "./n_dem_fine"
fuel_dir = "./n_fuel_fine"
output_tfrecord  = "./last_dance_128_all_iterval_gan_fine.tfrecord"

# 1) TFRecord 생성
write_tfrecord(
    output_path    = output_tfrecord,
    weather_txt_path= weather_txt_path,
    spread_dir     = spread_dir,
    terrain_dir    = terrain_dir,
    dem_dir= dem_dir,
    fuel_dir=fuel_dir,
    spread_size    = (128,128),
    terrain_size   = (128,128)   
)


In [ ]:
import tensorflow as tf


def parse_tfrecord(example_proto, image_size=128):
    # TFRecord에 저장된 feature 구성
    feature_description = {
        'prev_fire': tf.io.FixedLenFeature([], tf.string),
        'fire_1': tf.io.FixedLenFeature([], tf.string),
        'fire_2': tf.io.FixedLenFeature([], tf.string),
        'fire_3': tf.io.FixedLenFeature([], tf.string),
        'fire_4': tf.io.FixedLenFeature([], tf.string),
        'fire_5': tf.io.FixedLenFeature([], tf.string),
        'fire_6': tf.io.FixedLenFeature([], tf.string),
        'fire_7': tf.io.FixedLenFeature([], tf.string),
        'fire_8': tf.io.FixedLenFeature([], tf.string),
        'fire_9': tf.io.FixedLenFeature([], tf.string),
        'fire_10': tf.io.FixedLenFeature([], tf.string),
        'fire_11': tf.io.FixedLenFeature([], tf.string),
        'fire_12': tf.io.FixedLenFeature([], tf.string),
        'terrain_image': tf.io.FixedLenFeature([], tf.string),
        'dem_image': tf.io.FixedLenFeature([], tf.string),
        'fuel_image': tf.io.FixedLenFeature([], tf.string),
        'temp': tf.io.FixedLenFeature([], tf.string),
        'wind': tf.io.FixedLenFeature([], tf.string),
        'ignite_index': tf.io.FixedLenFeature([], tf.int64),
        'weather_index': tf.io.FixedLenFeature([], tf.int64),
    }
    features = tf.io.parse_single_example(example_proto, feature_description)

    # [1] Spread images (Grayscale, (256,256,1))
    def decode_and_reshape_spread(key):
        # 저장 시 float32로 정규화된 raw bytes가 저장되어 있다고 가정
        img = tf.io.decode_raw(features[key], tf.float32)
        return tf.reshape(img, [image_size, image_size, 1])
    
    #spread_4h = decode_and_reshape_spread('spread_4h_image')
    #spread_8h = decode_and_reshape_spread('spread_8h_image')
    spread_0 = decode_and_reshape_spread('prev_fire')
    spread_1 = decode_and_reshape_spread('fire_1')
    spread_2 = decode_and_reshape_spread('fire_2')
    spread_3 = decode_and_reshape_spread('fire_3')
    spread_4 = decode_and_reshape_spread('fire_4')
    spread_5 = decode_and_reshape_spread('fire_5')
    spread_6 = decode_and_reshape_spread('fire_6')
    spread_7 = decode_and_reshape_spread('fire_7')
    spread_8 = decode_and_reshape_spread('fire_8')
    spread_9 = decode_and_reshape_spread('fire_9')
    spread_10 = decode_and_reshape_spread('fire_10')
    spread_11 = decode_and_reshape_spread('fire_11')
    spread_12 = decode_and_reshape_spread('fire_12')
    #spread_seq = tf.stack([spread_6h, spread_12h], axis=0)  # (3,256,256,1)

    # [2] Terrain image (RGB, (256,256,3))
    terrain = tf.io.decode_raw(features['terrain_image'], tf.float32)
    terrain = tf.reshape(terrain, [image_size, image_size, 3])
    
    dem = decode_and_reshape_spread('dem_image')
    fuel = decode_and_reshape_spread('fuel_image')
    
    tnh = tf.io.decode_raw(features['temp'], tf.float32)
    tnh = tf.reshape(tnh,[12,2])
    wind = tf.io.decode_raw(features['wind'], tf.float32)
    wind = tf.reshape(wind,[12,2])
    '''t_h_t0 = tf.io.decode_raw(features['temp_hume_t1'], tf.float32)
    t_h_t0 = tf.reshape(t_h_t0, [4,1,2])

    t_h_t1 = tf.io.decode_raw(features['temp_hume_t2'], tf.float32)
    t_h_t1 = tf.reshape(t_h_t1, [4,1,2])

    t_h_t2 = tf.io.decode_raw(features['temp_hume_t3'], tf.float32)
    t_h_t2 = tf.reshape(t_h_t2, [4,1,2])

    ws_wd_t0 = tf.io.decode_raw(features['ws_wd_t1'], tf.float32)
    ws_wd_t0 = tf.reshape(ws_wd_t0, [4,1,2])

    ws_wd_t1 = tf.io.decode_raw(features['ws_wd_t2'], tf.float32)
    ws_wd_t1 = tf.reshape(ws_wd_t1, [4,1,2])

    ws_wd_t2 = tf.io.decode_raw(features['ws_wd_t3'], tf.float32)
    ws_wd_t2 = tf.reshape(ws_wd_t2, [4,1,2])'''



    # 추가적으로 인덱스도 추출할 수 있음
    ignite_index = features['ignite_index']
    weather_index = features['weather_index']
    
    # 최종적으로 모델 학습에 사용할 형태:  
    # Generator 입력: (terrain, weather_seq)  
    # Target (spread): spread_seq  
    # (추후 Discriminator에도 동일하게 사용)
    return (spread_0, spread_1,spread_2, spread_3, spread_4, spread_5,spread_6,spread_7,spread_8,spread_9,spread_10,spread_11,spread_12, terrain, dem, fuel, tnh, wind, ignite_index,weather_index)







def parse_tfrecord_v2(example_proto):
    # TFRecord에 저장된 feature 구성
    feature_description = {
        'target_fire_t1': tf.io.FixedLenFeature([], tf.string),
        'target_fire_t2': tf.io.FixedLenFeature([], tf.string),
        'target_fire_t3': tf.io.FixedLenFeature([], tf.string),
        'prev_fire': tf.io.FixedLenFeature([], tf.string),
        'terrain_image': tf.io.FixedLenFeature([], tf.string),
        'dem_image': tf.io.FixedLenFeature([], tf.string),
        'fuel_image': tf.io.FixedLenFeature([], tf.string),
        'temp': tf.io.FixedLenFeature([], tf.string),
        'hume': tf.io.FixedLenFeature([], tf.string),
        'ignite_index': tf.io.FixedLenFeature([], tf.int64),
        'weather_index': tf.io.FixedLenFeature([], tf.int64),
    }
    features = tf.io.parse_single_example(example_proto, feature_description)

    # [1] Spread images (Grayscale, (256,256,1))
    def decode_and_reshape_spread(key):
        # 저장 시 float32로 정규화된 raw bytes가 저장되어 있다고 가정
        img = tf.io.decode_raw(features[key], tf.float32)
        return tf.reshape(img, [128, 128, 1])
    
    #spread_4h = decode_and_reshape_spread('spread_4h_image')
    #spread_8h = decode_and_reshape_spread('spread_8h_image')
    spread_0h = decode_and_reshape_spread('prev_fire')
    spread_4h = decode_and_reshape_spread('target_fire_t1')
    spread_8h = decode_and_reshape_spread('target_fire_t2')
    spread_12h = decode_and_reshape_spread('target_fire_t3')
    #spread_seq = tf.stack([spread_6h, spread_12h], axis=0)  # (3,256,256,1)

    # [2] Terrain image (RGB, (256,256,3))
    terrain = tf.io.decode_raw(features['terrain_image'], tf.float32)
    terrain = tf.reshape(terrain, [256, 256, 3])
    
    dem = decode_and_reshape_spread('dem_image')
    fuel = decode_and_reshape_spread('fuel_image')
    

    t_h_t0 = tf.io.decode_raw(features['temp_hume_t1'], tf.float32)
    t_h_t0 = tf.reshape(t_h_t0, [4,1,2])

    t_h_t1 = tf.io.decode_raw(features['temp_hume_t2'], tf.float32)
    t_h_t1 = tf.reshape(t_h_t1, [4,1,2])

    t_h_t2 = tf.io.decode_raw(features['temp_hume_t3'], tf.float32)
    t_h_t2 = tf.reshape(t_h_t2, [4,1,2])

    ws_wd_t0 = tf.io.decode_raw(features['ws_wd_t1'], tf.float32)
    ws_wd_t0 = tf.reshape(ws_wd_t0, [4,1,2])

    ws_wd_t1 = tf.io.decode_raw(features['ws_wd_t2'], tf.float32)
    ws_wd_t1 = tf.reshape(ws_wd_t1, [4,1,2])

    ws_wd_t2 = tf.io.decode_raw(features['ws_wd_t3'], tf.float32)
    ws_wd_t2 = tf.reshape(ws_wd_t2, [4,1,2])



    # 추가적으로 인덱스도 추출할 수 있음
    ignite_index = features['ignite_index']
    weather_index = features['weather_index']
    
    # 최종적으로 모델 학습에 사용할 형태:  
    # Generator 입력: (terrain, weather_seq)  
    # Target (spread): spread_seq  
    # (추후 Discriminator에도 동일하게 사용)
    return (spread_0h, spread_4h,spread_8h, spread_12h, terrain, dem, fuel, t_h_t0, t_h_t1, t_h_t2,ws_wd_t0, ws_wd_t1, ws_wd_t2, ignite_index,weather_index)



def get_dataset_size(dataset):
    """dataset의 전체 길이(샘플 개수)를 파이썬 루프로 센다."""
    count = 0
    for _ in dataset:
        count += 1
    return count

def create_all_dataset(tfrecord_files):
    """
    TFRecord -> parse -> 하나의 Dataset 반환
    (아직 shuffle이나 batch 안 함)
    """
    ds = tf.data.TFRecordDataset(tfrecord_files)
    ds = ds.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    return ds

def train_test_split_by_length(tfrecord_files, train_ratio=0):
    """
    1) 전체 dataset 길이 구하기
    2) shuffle(전 샘플), then take/skip
    """
    # 1) 첫 번째 pass로 dataset 길이 구함
    ds_for_count = create_all_dataset(tfrecord_files)
    dataset_size = get_dataset_size(ds_for_count)
    print("Dataset size:", dataset_size)

    # 2) 두 번째 pass: 실제 dataset
    ds_full = create_all_dataset(tfrecord_files)
    ds_full = ds_full.shuffle(dataset_size, reshuffle_each_iteration=False)

    train_size = int(dataset_size * train_ratio)

    # take(train_size), skip(train_size)
    train_ds = ds_full.take(train_size)
    test_ds  = ds_full.skip(train_size)
    return train_ds, test_ds


# 사용 예시
tfrecord_files = ["./last_dance_128_all_iterval_gan_fine.tfrecord"]
#tfrecord_files = ["./ds_128_all_gan.tfrecord"]
#tfrecord_files = ["./train_ds_128_all_iterval_gan.tfrecord"]
train_ds, test_ds = train_test_split_by_length(tfrecord_files, train_ratio=0.8)



def filter_ignite_weather(dataset, ign_min, ign_max, w_min, w_max):
    return dataset.filter(
        lambda  spread_0,spread_1,spread_2,spread_3, spread_4,spread_5,spread_6,spread_7, spread_8,spread_9,spread_10,spread_11,spread_12, terrain, dem, fuel, tnh,wind, ig, w: 
            (ig >= ign_min) & (ig <= ign_max) &
            (w  >= w_min)   & (w  <= w_max)
    )
# train_tiny = filter_ignite_weather(train_ds, 1,200, 1,10).batch(16)
#train_small = filter_ignite_weather(train_ds, 1,200, 1,30).batch(16)
# train_mid = filter_ignite_weather(train_ds, 1,300, 1,45).batch(16)
train_big = filter_ignite_weather(train_ds, 1,300, 1,70).batch(16)
# test_tiny = filter_ignite_weather(test_ds, 1,200, 1,10).batch(16)
#test_small  = filter_ignite_weather(test_ds, 1,200, 1,30).batch(16)
# test_mid  = filter_ignite_weather(test_ds, 1,300, 1,45).batch(16)
test_big  = filter_ignite_weather(test_ds, 1,300, 1,70).batch(16)
# test_test = filter_ignite_weather(test_ds, 1,300, 1,70).batch(16)


In [ ]:
import tensorflow as tf

def _bytes_feature(value: bytes):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _int64_feature(value: int):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def write_subdataset_to_tfrecord(dataset, output_path):

    writer = tf.io.TFRecordWriter(output_path)
    sample_count = 0

    for batch in dataset:
        # batch = (terrain_b, weather_seq_b, spread_seq_b, ign_b, w_b)
        spread_0,spread_1,spread_2,spread_3, spread_4,spread_5,spread_6,spread_7, spread_8,spread_9,spread_10,spread_11,spread_12, terrain, dem, fuel, tnh,wind, ii, wi = batch

        batch_size = terrain.shape[0]  # B
        for i in range(batch_size):
            # ---------------------
            # 1) 텐서 하나씩 numpy 변환
            # ---------------------
            terrain_np    = terrain[i].numpy()           # (256,256,3)
            dem_np = dem[i].numpy()
            fuel_np = fuel[i].numpy()
            #th_0h_np= weather_seq_b[i].numpy()       # (3,256,256,4)
            spread_0h_np = spread_0[i].numpy()        # (3,256,256,1)
            spread_1h_np = spread_1[i].numpy()        # (3,256,256,1)
            spread_2h_np = spread_2[i].numpy()        # (3,256,256,1)
            spread_3h_np = spread_3[i].numpy()        # (3,256,256,1)
            spread_4h_np = spread_4[i].numpy()        # (3,256,256,1)
            spread_5h_np = spread_5[i].numpy()        # (3,256,256,1)
            spread_6h_np = spread_6[i].numpy()        # (3,256,256,1)
            spread_7h_np = spread_7[i].numpy()        # (3,256,256,1)
            spread_8h_np = spread_8[i].numpy()        # (3,256,256,1)
            spread_9h_np = spread_9[i].numpy()        # (3,256,256,1)
            spread_10h_np = spread_10[i].numpy()        # (3,256,256,1)
            spread_11h_np = spread_11[i].numpy()        # (3,256,256,1)
            spread_12h_np = spread_12[i].numpy()        # (3,256,256,1)
            ign_val       = ii[i].numpy()               # scalar
            we_val        = wi[i].numpy()                # scalar


            #spread_bytes = spread_seq_np.tobytes()
            spread_0h = spread_0h_np.tobytes()        # (3,256,256,1)
            spread_1h = spread_1h_np.tobytes()        # (3,256,256,1)
            spread_2h = spread_2h_np.tobytes()        # (3,256,256,1)
            spread_3h = spread_3h_np.tobytes()        # (3,256,256,1)
            spread_4h = spread_4h_np.tobytes()        # (3,256,256,1)
            spread_5h = spread_5h_np.tobytes()        # (3,256,256,1)
            spread_6h = spread_6h_np.tobytes()        # (3,256,256,1)
            spread_7h = spread_7h_np.tobytes()        # (3,256,256,1)
            spread_8h = spread_8h_np.tobytes()        # (3,256,256,1)
            spread_9h = spread_9h_np.tobytes()        # (3,256,256,1)
            spread_10h = spread_10h_np.tobytes()        # (3,256,256,1)
            spread_11h = spread_11h_np.tobytes()        # (3,256,256,1)
            spread_12h = spread_12h_np.tobytes()        # (3,256,256,1)
            terrain_bytes     = terrain_np.tobytes()
            dem_bytes = dem_np.tobytes()
            fuel_bytes = fuel_np.tobytes()
            tnh_np = tnh[i].numpy()
            wind_np =wind[i].numpy()
            tnh_byte = tnh_np.tobytes()
            wind_byte = wind_np.tobytes()

            # ---------------------
            # 5) TF Example에 담기
            # ---------------------
            feature_dict = {
                #'spread_6h_image': _bytes_feature(spread_6h_bytes),
                'spread_0h': _bytes_feature(spread_0h),
                'spread_1h': _bytes_feature(spread_1h),
                'spread_2h': _bytes_feature(spread_2h),
                'spread_3h': _bytes_feature(spread_3h),
                'spread_4h': _bytes_feature(spread_4h),
                'spread_5h': _bytes_feature(spread_5h),
                'spread_6h': _bytes_feature(spread_6h),
                'spread_7h': _bytes_feature(spread_7h),
                'spread_8h': _bytes_feature(spread_7h),
                'spread_9h': _bytes_feature(spread_8h),
                'spread_10h': _bytes_feature(spread_10h),
                'spread_11h': _bytes_feature(spread_11h),
                'spread_12h': _bytes_feature(spread_12h),
                'terrain_image':   _bytes_feature(terrain_bytes),
                'dem_image': _bytes_feature(dem_bytes),
                'fuel_image': _bytes_feature(fuel_bytes),
                'tnh':  _bytes_feature(tnh_byte),
                'wind':  _bytes_feature(wind_byte),
                #'temp_6h':  _bytes_feature(temp_6h_bytes),
                #'hume_6h':  _bytes_feature(hume_6h_bytes),
                #'ws_6h':    _bytes_feature(ws_6h_bytes),
                #'wd_6h':    _bytes_feature(wd_6h_bytes),
                'ignite_index':  _int64_feature(int(ign_val)),
                'weather_index': _int64_feature(int(we_val)),
            }

            example = tf.train.Example(features=tf.train.Features(feature=feature_dict))
            writer.write(example.SerializeToString())
            sample_count += 1

    writer.close()
    print(f"[INFO] {output_path} saved, total {sample_count} samples")



In [ ]:
# 가정: 앞서 정의한 train_small, train_mid, ... 등 dataset이 있음

#write_subdataset_to_tfrecord(train_small, "train_small_128_all_origin_modi.tfrecord")
# write_subdataset_to_tfrecord(train_mid,   "train_mid_256_interval.tfrecord")
write_subdataset_to_tfrecord(train_big,   "train_big_128_all_origin_fine.tfrecord")

#write_subdataset_to_tfrecord(test_small,  "test_small_128_all_origin_modi.tfrecord")
# write_subdataset_to_tfrecord(test_mid,    "test_mid_256_interval.tfrecord")
write_subdataset_to_tfrecord(test_big,    "test_big_128_all_origin_fine.tfrecord")

#write_subdataset_to_tfrecord(train_tiny,    "train_tiny_256_interval.tfrecord")
#write_subdataset_to_tfrecord(test_tiny,    "test_tiny_256_interval.tfrecord")
# write_subdataset_to_tfrecord(test_test, "test_128_56ea.tfrecord")

In [ ]:



def make_subdataset(
    dataset,
    ignite_min, ignite_max,
    weather_min, weather_max,
    batch_size=16,
    shuffle_buffer=1000,
    drop_remainder=False
):
    # filter
    ds_filtered = dataset.filter(
        lambda  *rest,ig, w: (
            (ig >= ignite_min) & (ig <= ignite_max) &
            (w  >= weather_min) & (w  <= weather_max)
        )
    )
    # shuffle, batch, prefetch
    ds_filtered = ds_filtered.shuffle(shuffle_buffer)
    ds_filtered = ds_filtered.batch(batch_size, drop_remainder=drop_remainder)
    ds_filtered = ds_filtered.prefetch(tf.data.AUTOTUNE)
    return ds_filtered

def create_three_subdatasets(
    dataset,
    spread_shape=(256,256,1),
    terrain_shape=(256,256,3),
    weather_shape=(256,256,1),
    batch_size=16,
    shuffle_buffer=1000
):
    # 1) 전체 데이터 불러오기
    dataset = dataset
    
    # 2) 파싱
    dataset = dataset.map(
        lambda x: parse_example_function(
            x,
            spread_shape=spread_shape,
            terrain_shape=terrain_shape,
            weather_shape=weather_shape
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    
    # 3) 각각의 subdataset 필터링
    # small: ignite=[1..100], weather=[1..20]
    subdataset_small = make_subdataset(
        dataset,
        ignite_min=1, ignite_max=100,
        weather_min=1, weather_max=20,
        batch_size=batch_size,
        shuffle_buffer=shuffle_buffer
    )

    # mid: ignite=[1..200], weather=[1..50]
    subdataset_mid = make_subdataset(
        dataset,
        ignite_min=1, ignite_max=200,
        weather_min=1, weather_max=50,
        batch_size=batch_size,
        shuffle_buffer=shuffle_buffer
    )

    # full: ignite=[1..300], weather=[1..70]
    subdataset_full = make_subdataset(
        dataset,
        ignite_min=1, ignite_max=300,
        weather_min=1, weather_max=70,
        batch_size=batch_size,
        shuffle_buffer=shuffle_buffer
    )

    return subdataset_small, subdataset_mid, subdataset_full
